In [21]:
import pandas as pd
import numpy as np
from glob import glob
import re
import nltk
import os, re
from pathlib import Path
import re
from pathlib import Path

import pandas as pd
import nltk
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize, word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('tagsets')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package tagsets to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/samanthaasefi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date

True

In [22]:
AUTHOR_MAP = {
    'luxemburg_reform-or-revolution': 'Rosa Luxemburg',
    'mao_on-guerrilla-warfare': 'Mao Zedong',
    'marx_communist-manifesto': 'Karl Marx & Friedrich Engels',
    'fourier_selections': 'Charles Fourier'
}

TITLE_MAP = {
    'luxemburg_reform-or-revolution': 'Reform or Revolution',
    'mao_on-guerrilla-warfare': 'On Guerrilla Warfare',
    'marx_communist-manifesto': 'The Communist Manifesto',
    'fourier_selections': 'Selections from his Writings'
}

In [23]:
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

In [24]:
CORPUS_DIR = Path("corpus/books")

In [25]:
def parse_chap_num(filepath: Path):
    """
    Extract chapter number from filenames like:
    ch00_intro.txt, ch01.txt, ch12.txt
    """
    m = re.match(r"ch(\d+)", filepath.stem.lower())
    if m:
        return int(m.group(1))
    return None

In [26]:
def parse_raw_txt(filepath: Path):
    text = filepath.read_text(encoding="utf-8")
    soup = BeautifulSoup(text, "html.parser")
    
    source_url_tag = soup.find("source_url")
    page_title_tag = soup.find("page_title")
    chapter_title_tag = soup.find("chapter_title")
    
    source_url = source_url_tag.get_text(strip=True) if source_url_tag else None
    page_title = page_title_tag.get_text(strip=True) if page_title_tag else None
    chapter_title = chapter_title_tag.get_text(strip=True) if chapter_title_tag else None
    
    paras = []
    for p in soup.find_all("para"):
        para_num = int(p.get("id"))
        para_text = p.get_text(" ", strip=True)
        paras.append((para_num, para_text))
    
    return {
        "source_url": source_url,
        "page_title": page_title,
        "chapter_title": chapter_title,
        "paragraphs": paras
    }

In [27]:
def is_real_content(text: str) -> bool:
    if not text:
        return False
    
    txt = text.strip()
    
    # source/bibliographic line
    if txt.startswith("Source:"):
        return False
    
    # archive/footer line
    if "Archive" in txt:
        return False
    
    # author lifespan line, e.g. Charles Fourier (1772-1837)
    if re.fullmatch(r".*\(\d{4}\s*-\s*\d{4}\)", txt):
        return False
    
    return True

# Register and Chunk

In [28]:
# # Register and Chunk

library_rows = []
doc_rows = []

for book_dir in sorted(CORPUS_DIR.iterdir()):
    if not book_dir.is_dir():
        continue

    book_id = book_dir.name

    for filepath in sorted(book_dir.glob("*.txt")):
        chap_num = parse_chap_num(filepath)
        parsed = parse_raw_txt(filepath)

        # LIB row = one chapter file
        library_rows.append({
            'book_id': book_id,
            'chap_num': chap_num,
            'author': AUTHOR_MAP.get(book_id),
            'book_title': TITLE_MAP.get(book_id),
            'filepath': str(filepath),
            'source_url': parsed['source_url'],
            'page_title': parsed['page_title'],
            'chapter_title': parsed['chapter_title'],
            'n_paragraphs_raw': len(parsed['paragraphs'])
        })

        # DOC rows = one paragraph per row
        for para_num, para_text in parsed['paragraphs']:
            if is_real_content(para_text):
                doc_rows.append({
                    'book_id': book_id,
                    'chap_num': chap_num,
                    'para_num': para_num,
                    'author': AUTHOR_MAP.get(book_id),
                    'book_title': TITLE_MAP.get(book_id),
                    'para_str': para_text
                })

LIB = pd.DataFrame(library_rows).sort_values(['book_id', 'chap_num']).reset_index(drop=True)
DOC = pd.DataFrame(doc_rows).sort_values(['book_id', 'chap_num', 'para_num']).reset_index(drop=True)

In [40]:
LIB.head()

,book_id,chap_num,author,book_title,filepath,source_url,page_title,chapter_title,n_paragraphs_raw
0,fourier_selections,1,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch01.txt,https://www.marxists.org/reference/archive/fou...,"Charles Fourier, Selections from his Writings",Of the Role of the Passions,41
1,fourier_selections,2,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch02.txt,https://www.marxists.org/reference/archive/fou...,"Charles Fourier, Selections from his Writings",Of Education,23
2,fourier_selections,3,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch03.txt,https://www.marxists.org/reference/archive/fou...,Selection from Charles Fourier,“Universal Harmony”,10
3,fourier_selections,4,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch04.txt,https://www.marxists.org/reference/archive/fou...,Selection from Charles Fourier,“Letter to the High Judge”,43
4,fourier_selections,5,Charles Fourier,Selections from his Writings,corpus/books/fourier_selections/ch05.txt,https://www.marxists.org/reference/archive/fou...,Selection from Charles Fourier,“Indices and Methods which led to the Discovery”,34


In [29]:
DOC = DOC.set_index(['book_id', 'chap_num', 'para_num']).sort_index()

In [39]:
DOC.head()

author  \
book_id            chap_num para_num                    
fourier_selections 1        3         Charles Fourier   
                            4         Charles Fourier   
                            5         Charles Fourier   
                            6         Charles Fourier   
                            7         Charles Fourier   

                                                        book_title  \
book_id            chap_num para_num                                 
fourier_selections 1        3         Selections from his Writings   
                            4         Selections from his Writings   
                            5         Selections from his Writings   
                            6         Selections from his Writings   
                            7         Selections from his Writings   

                                                                               para_str  
book_id            chap_num para_num                                                     
fourier_selections 1        3         All those philosophical whims called duties ha...  
                            4         The learned world is wholly imbued with a doct...  
                            5         Morality teaches man to be at war with himself...  
                            6         It is true that these impulses entice us only ...  
                            7         The passions, believed to be the enemies of co...

# Tokenize and Annotate

In [30]:
def tokenize(doc_df, OHCO=OHCO, remove_pos_tuple=False, ws=False):

    # Paragraphs -> Sentences
    df = (
        doc_df['para_str']
        .apply(lambda x: pd.Series(nltk.sent_tokenize(x)))
        .stack()
        .to_frame(name='sent_str')
    )

    # Sentences -> Tokens
    def tokenize_sentence(x):
        if ws:
            toks = nltk.WhitespaceTokenizer().tokenize(x)
        else:
            toks = nltk.word_tokenize(x)
        return pd.Series(nltk.pos_tag(toks))

    df = (
        df['sent_str']
        .apply(tokenize_sentence)
        .stack()
        .to_frame(name='pos_tuple')
    )

    # Pull token/POS out of tuple
    df['token_str'] = df['pos_tuple'].apply(lambda x: x[0])
    df['pos'] = df['pos_tuple'].apply(lambda x: x[1])

    if remove_pos_tuple:
        df = df.drop(columns='pos_tuple')

    # Set index names to OHCO
    df.index.names = OHCO

    return df

In [31]:
TOKEN = tokenize(DOC, OHCO=OHCO, remove_pos_tuple=False, ws=False)


In [32]:
doc_meta = (
    DOC.reset_index()[['book_id', 'chap_num', 'author', 'book_title']]
    .drop_duplicates()
)

TOKEN = TOKEN.reset_index()

TOKEN['term_str'] = (
    TOKEN['token_str']
    .str.lower()
    .str.replace(r"[^a-z']+", "", regex=True)
)

TOKEN = TOKEN[TOKEN['term_str'] != ""].copy()
TOKEN = TOKEN.set_index(OHCO).sort_index()


In [37]:
TOKEN.head()

pos_tuple  \
book_id            chap_num para_num sent_num token_num                        
fourier_selections 1        3        0        0                   (All, PDT)   
                                              1                  (those, DT)   
                                              2          (philosophical, JJ)   
                                              3                 (whims, NNS)   
                                              4                (called, VBN)   

                                                             token_str  pos  \
book_id            chap_num para_num sent_num token_num                       
fourier_selections 1        3        0        0                    All  PDT   
                                              1                  those   DT   
                                              2          philosophical   JJ   
                                              3                  whims  NNS   
                                              4                 called  VBN   

                                                              term_str  
book_id            chap_num para_num sent_num token_num                 
fourier_selections 1        3        0        0                    all  
                                              1                  those  
                                              2          philosophical  
                                              3                  whims  
                                              4                 called

In [38]:
TOKEN[TOKEN.pos.str.match('^JJ')]

pos_tuple  \
book_id                  chap_num para_num sent_num token_num                        
fourier_selections       1        3        0        2          (philosophical, JJ)   
                                                    66            (invariable, JJ)   
                                  4        0        1                (learned, JJ)   
                                                    15                (mortal, JJ)   
                                                    18             (passional, JJ)   
...                                                                            ...   
marx_communist-manifesto 4        14       0        31                  (euch, JJ)   
                                                    38               (correct, JJ)   
                                  15       1        15                  (last, JJ)   
                                                    25              (official, JJ)   
                                           2        13              (original, JJ)   

                                                                   token_str  \
book_id                  chap_num para_num sent_num token_num                  
fourier_selections       1        3        0        2          philosophical   
                                                    66            invariable   
                                  4        0        1                learned   
                                                    15                mortal   
                                                    18             passional   
...                                                                      ...   
marx_communist-manifesto 4        14       0        31                  euch   
                                                    38               correct   
                                  15       1        15                  last   
                                                    25              official   
                                           2        13              original   

                                                              pos  \
book_id                  chap_num para_num sent_num token_num       
fourier_selections       1        3        0        2          JJ   
                                                    66         JJ   
                                  4        0        1          JJ   
                                                    15         JJ   
                                                    18         JJ   
...                                                            ..   
marx_communist-manifesto 4        14       0        31         JJ   
                                                    38         JJ   
                                  15       1        15         JJ   
                                                    25         JJ   
                                           2        13         JJ   

                                                                    term_str  
book_id                  chap_num para_num sent_num token_num                 
fourier_selections       1        3        0        2          philosophical  
                                                    66            invariable  
                                  4        0        1                learned  
                                                    15                mortal  
                                                    18             passional  
...                                                                      ...  
marx_communist-manifesto 4        14       0        31                  euch  
                                                    38               correct  
                                  15       1        15                  last  
                                                    25              official  
                                           2        13             

## Building vocab

In [33]:
VOCAB = (
    TOKEN.reset_index()
    .groupby('term_str')
    .agg(
        n=('term_str', 'size')
    )
    .sort_values('n', ascending=False)
)

In [35]:
OUTDIR = Path("corpus/f2")
OUTDIR.mkdir(parents=True, exist_ok=True)

In [36]:
LIB.to_csv("corpus/f2/LIB.csv", index=False)
DOC.reset_index().to_csv("corpus/f2/DOC.csv", index=False)
TOKEN.reset_index().to_csv("corpus/f2/TOKEN.csv", index=False)
VOCAB.reset_index().to_csv("corpus/f2/VOCAB.csv", index=False)